# Behavior Modeling API: InterSim Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. InterSim is one of the default behavior models integrated into Tactics2D.

Original paper: [InterSim: Interactive Traffic Simulation via Explicit Relation Modeling](https://arxiv.org/abs/2210.11445)
Original code: [Tsinghua-MARS-Lab/InterSim](https://github.com/Tsinghua-MARS-Lab/InterSim)

InterSim's claim is that what matters is not that two vehicles are close, but **which of them has to give way**. It scans for imminent conflicts, resolves each into a directed relation (influencer to reactor), and lets the reactor brake while the influencer keeps its path. It is a rule-based model - the learned relation arbiter and the marginal predictor stay closed in this port - so no checkpoint is needed.


## Environment Setup

Please install Tactics2D (`pip install 'tactics2d[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details. InterSim needs no checkpoint: the model is built from its configuration alone.

!!! info "Two capabilities are configured but closed"
    `relation_mode="nn"`, the learned M2I direction arbiter, needs a checkpoint and a caller-supplied `decider`; the marginal predictor must stay off (`use_marginal_model=False`). Both raise on construction if enabled.


## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when parsing it. Adjust the paths below to match your local data layout.

| Dataset | Role here | Rate | Map |
|---------|-----------|------|-----|
| **WOMD** | the scene the four behavior demos share, so their numbers can be read side by side | 10 Hz | per-scenario, from the tfrecord |
| **inD** | off-domain: a German urban junction, recorded by a different group | 25 Hz | Lanelet2 `.osm` |
| **nuPlan** | off-domain: a city-scale map, one log carrying every lane of Boston | 20 Hz | `.gpkg` |

!!! warning "The model runs on a fixed 100 ms lattice"
    Every behavior model lays a scenario out on a fixed step - InterSim's is `step_ms = 100`, and its per-agent pose arrays are indexed on it. A log recorded at another rate is **resampled onto that lattice automatically** by the runner (`tactics2d.behavior.rolling_utils.to_lattice`); fed as-is, several of its frames would land on one index and the pose array would end up holding a fraction of the recording.

    No other 10 Hz dataset is set up here, so every off-domain example below is also an off-rate one. That is a gap in the data on hand, not a step the demos skipped.


## Use InterSim for Behavior Generation

Both usages below go through the same public API; the notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` -> `(participants, time_range)`; `parser.parse_map(...)` -> `Map` |
| **Behavior model** | `InterSimBehaviorModel(config)` -> `.predict(...)`, `.plan(...)`, `.rollout(...)` |
| **Rendering** | `BEVCamera` + `MatplotlibRenderer`, driven through `tutorial_common.render_replay_animation` |


In [1]:
import warnings

warnings.filterwarnings("ignore")

import logging
from pathlib import Path

logging.basicConfig(level=logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np

import tutorial_common
from tactics2d.behavior import InterSimBehaviorModel, InterSimConfig
from tactics2d.behavior.rolling_utils import to_lattice
from tactics2d.dataset_parser import LevelXParser, NuPlanParser, WOMDParser
from tactics2d.map.map_config import IND_MAP_CONFIG
from tactics2d.map.parser import OSMParser
from tactics2d.participant.element import Vehicle
from tactics2d.participant.trajectory import State, Trajectory

pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
tutorial_common.apply_notebook_style()

In [3]:
# InterSim model config (the reproduction defaults; the learned predictors stay closed)
INTERSIM_CFG = InterSimConfig(cruise_speed=8.0)

# Prediction / display constants. The perception range and view window are shared
# with the other behavior demos and live in tutorial_common.
WARMUP_MS = INTERSIM_CFG.planning_warmup_steps * INTERSIM_CFG.step_ms
PLAYBACK_STEP = 1  # render every N-th simulated step (1 = real time at 10 fps)

print("relation mode:", INTERSIM_CFG.relation_mode)
print("step time:", INTERSIM_CFG.step_ms, "ms")
print(
    "planning horizon:",
    INTERSIM_CFG.horizon_steps,
    "steps =",
    INTERSIM_CFG.horizon_steps * INTERSIM_CFG.dt,
    "s",
)
print("interaction distance:", INTERSIM_CFG.interaction_distance, "m")
print(
    "closed-loop cadence: warmup",
    INTERSIM_CFG.planning_warmup_steps,
    "frames, replan every",
    INTERSIM_CFG.planning_interval,
    "frames",
)

# The shared take-over protocol lives in tutorial_common; this is where its file is.
WOMD_FOLDER = f"../../../data/womd/uncompressed/{tutorial_common.COMPARISON_SPLIT}"

relation mode: directed
step time: 100 ms
planning horizon: 80 steps = 8.0 s
interaction distance: 40.0 m
closed-loop cadence: warmup 11 frames, replan every 10 frames


## Take-Over Usage

`predict()` replaces the future of **one** vehicle and leaves everybody else on the trajectory the log recorded. It is the call all four Tactics2D behavior models share, and the one the cross-model comparison is run on.

```python
plan = model.predict(participants, map_, frame, agent_ids=[ego_id])
```

`frame` is the newest frame the model conditions on, in milliseconds, and the return value is `{agent_id: Trajectory}`. The prediction is scored against the vehicle's recorded future with `tutorial_common.displacement_errors`.

### Step 1: Select the Vehicle

This demo uses the shared comparison vehicle - the same junction and the same vehicle the LimSim and SMART demos take over - so the three numbers can be read side by side.

### Step 2: Predict, and Score Against the Recorded Future

Scored twice: over the shared 2 s horizon every behavior demo covers, and over InterSim's own 8 s.

### Step 3: Render with BEVCamera and MatplotlibRenderer

The purple gradient is the path the vehicle has already driven, the green one its plan. Everything outside the modelled set keeps its recorded motion.


In [4]:
takeover_parser = WOMDParser()
takeover_participants, takeover_time_range = takeover_parser.parse_trajectory(
    tutorial_common.COMPARISON_SCENARIO, file=tutorial_common.COMPARISON_FILE, folder=WOMD_FOLDER
)
takeover_participants = to_lattice(takeover_participants, INTERSIM_CFG.step_ms)
takeover_map = takeover_parser.parse_map(
    tutorial_common.COMPARISON_SCENARIO, file=tutorial_common.COMPARISON_FILE, folder=WOMD_FOLDER
)

takeover_ego = tutorial_common.COMPARISON_EGO
ego = takeover_participants[takeover_ego]
ego.color = tutorial_common.EGO_COLOR
# Set the recorded future aside before anything overwrites it.
truth = tutorial_common.recorded_future(ego.trajectory, tutorial_common.COMPARISON_FRAME_MS)

takeover_model = InterSimBehaviorModel(INTERSIM_CFG)
plan = takeover_model.predict(
    takeover_participants,
    takeover_map,
    frame=tutorial_common.COMPARISON_FRAME_MS,
    agent_ids=[takeover_ego],
)[takeover_ego]
ade, fde, matched = tutorial_common.displacement_errors(
    plan, truth, tutorial_common.COMPARISON_HORIZON_STEPS
)
own_ade, own_fde, own_matched = tutorial_common.displacement_errors(plan, truth)
print(
    f"ego: {takeover_ego}  |  predicted {len(plan.frames)} steps, "
    f"{min(plan.frames)}-{max(plan.frames)} ms"
)
print(f"  vs the recorded future:  ADE@2s {ade:.3f} m  FDE@2s {fde:.3f} m  ({matched} steps)")
print(
    f"                           ADE@8s {own_ade:.3f} m  FDE@8s {own_fde:.3f} m  "
    f"({own_matched} steps)"
)

# Resample the plan onto the recorded frame grid.
plan_frames = sorted(plan.frames)
ego.trajectory._history_states = {
    frame: state
    for frame, state in ego.trajectory.history_states.items()
    if frame <= tutorial_common.COMPARISON_FRAME_MS
}
ego.trajectory._frames = sorted(ego.trajectory._history_states)
takeover_plan = {}
for frame in sorted(truth.frames):
    nearest = min(plan_frames, key=lambda other: abs(other - frame))
    if abs(nearest - frame) > 50:
        continue
    state = plan.get_state(nearest)
    ego.trajectory.add_state(State(frame=frame, x=state.x, y=state.y, heading=state.heading))
    takeover_plan[frame] = (state.x, state.y)

playback_frames = [f for f in ego.trajectory.frames if f <= takeover_time_range[1]]
ani_womd_2_takeover = tutorial_common.render_replay_animation(
    takeover_participants,
    takeover_map,
    playback_frames,
    takeover_ego,
    plans={
        tutorial_common.COMPARISON_FRAME_MS: [
            (f, x, y) for f, (x, y) in sorted(takeover_plan.items())
        ]
    },
    fps=1000.0 / INTERSIM_CFG.step_ms,
    title_prefix="InterSim take-over",
)
ani_womd_2_takeover

ego: 8  |  predicted 80 steps, 1200-9100 ms
  vs the recorded future:  ADE@2s 2.726 m  FDE@2s 6.986 m  (20 steps)
                           ADE@8s 35.726 m  FDE@8s 82.118 m  (79 steps)


## Closed-Loop Usage

`rollout()` is the other mode: it replays the whole scenario. The ego and the conflict set it grows are re-planned and committed together, replanning once per second, while the remaining vehicles keep their recorded motion. The runner returns metrics and one pose array per participant rather than an animation, so the last two steps below turn those arrays back into something the camera can draw.

### Step 1: Select the Ego Vehicle

`tutorial_common.select_ego` picks a vehicle present before the warm-up frame that survives into the second half of the scenario; passing an explicit `ego_id` is just as valid.

### Step 2: Predict Directed Relations

`plan()` is the richer sibling of `predict()`: it returns the directed relations it found, the action it assigned each agent, and the trajectories. `plan_interaction_scene` below plans the ego first, reads the conflict set out of that result, and then plans the ego together with it.

### Step 3: Visualize the Directed Relations

An arrow runs from the influencer to the reactor, so its head points at the vehicle that has to give way.

### Step 4: Render with BEVCamera and MatplotlibRenderer

The runner commits its plans into the pose arrays but never returns them, so `collect_ego_plans` re-derives the ego's plan at each planning frame by asking the same model on the simulated state.


In [5]:
def frame_at_or_after(participant, frame_ms):
    """Return the first frame of the trajectory at or after ``frame_ms``."""
    frames = participant.trajectory.frames
    later = [f for f in frames if f >= frame_ms]
    return later[0] if later else frames[-1]

In [6]:
def plan_interaction_scene(model, participants, map_, frame, ego_id):
    """Plan the ego's interaction scene (ego plus the vehicles it conflicts with)."""
    probe = model.plan(participants, map_, frame, agent_ids=[ego_id])
    scene_ids = [agent_id for agent_id in probe.scene_agent_ids if agent_id in participants]
    return model.plan(participants, map_, frame, agent_ids=scene_ids)

In [7]:
def planned_xy(trajectory):
    states = [trajectory.get_state(frame) for frame in trajectory.frames]
    return np.asarray([[state.x, state.y] for state in states])


def history_xy(participant, frame):
    frames = sorted(f for f in participant.trajectory.history_states if f <= frame)
    if not frames:
        return np.zeros((0, 2))
    return np.asarray([participant.trajectory.get_state(f).location for f in frames])


def pose_at(participants, agent_id, frame):
    """Position of an agent, or None when its track has no state at that frame."""
    if agent_id not in participants:
        return None
    if not participants[agent_id].trajectory.has_state(frame):
        return None
    return np.asarray(participants[agent_id].trajectory.get_state(frame).location)


def plot_directed_relations(
    participants, map_, result, ego_id, frame, radius=30.0, title=None, save_to=None
):
    ego_current = pose_at(participants, ego_id, frame)

    def inside_window(point):
        return abs(point[0] - ego_current[0]) <= radius and abs(point[1] - ego_current[1]) <= radius

    def in_neighbourhood(agent_id):
        pose = pose_at(participants, agent_id, frame)
        return pose is not None and inside_window(pose)

    def clipped_to_window(points):
        mask = np.array([inside_window(point) for point in points])
        return np.where(mask[:, None], points, np.nan)

    # Keep every ego-centric relation, even for agents with no planned trajectory.
    window_relations = [
        edge for edge in result.relations if ego_id in (edge.influencer, edge.reactor)
    ]

    # Only the agents inside the view are labelled; the arrow of a relation whose
    # influencer sits further away is still drawn and simply clipped by the axes.
    related_ids = {
        agent_id
        for edge in window_relations
        for agent_id in (edge.influencer, edge.reactor)
        if in_neighbourhood(agent_id)
    }
    related_ids.add(ego_id)
    related_ids = sorted(related_ids, key=str)
    color_by_agent = {aid: plt.cm.tab10(index % 10) for index, aid in enumerate(related_ids)}

    figure, axes = plt.subplots(figsize=(7.5, 7.5))
    for lane in map_.lanes.values():
        for side in (lane.left_side, lane.right_side):
            if side is None or side.is_empty:
                continue
            xy = np.asarray(side.coords)
            axes.plot(xy[:, 0], xy[:, 1], color="#d4d4d4", linewidth=0.7, alpha=0.8, zorder=1)

    for agent_id in related_ids:
        color = color_by_agent[agent_id]
        history = clipped_to_window(history_xy(participants[agent_id], frame))
        if len(history) >= 2:
            axes.plot(
                history[:, 0],
                history[:, 1],
                color=color,
                linewidth=1.0,
                linestyle=":",
                alpha=0.9,
                zorder=5,
            )
        if agent_id in result.trajectories:
            planned = clipped_to_window(planned_xy(result.trajectories[agent_id]))
            axes.plot(
                planned[:, 0],
                planned[:, 1],
                color=color,
                linewidth=3.4 if agent_id == ego_id else 1.8,
                alpha=1.0 if agent_id == ego_id else 0.75,
                zorder=8,
            )
        state = pose_at(participants, agent_id, frame)
        axes.plot(state[0], state[1], marker="o", color=color, markersize=4, zorder=15)
        action = result.actions.get(agent_id, "?")
        label = (
            ("%s (ego): %s" % (agent_id, action))
            if agent_id == ego_id
            else ("%s: %s" % (agent_id, action))
        )
        axes.annotate(
            label,
            xy=(state[0], state[1]),
            xytext=(0, 11),
            textcoords="offset points",
            color=color,
            fontsize=8,
            fontweight="bold",
            ha="center",
            bbox={"facecolor": "white", "edgecolor": color, "alpha": 0.9, "pad": 1.2},
            zorder=30,
        )

    for edge in window_relations:
        start = pose_at(participants, edge.influencer, frame)
        end = pose_at(participants, edge.reactor, frame)
        if start is None or end is None:
            continue
        direction = end - start
        length = float(np.linalg.norm(direction))
        if length < 1e-3:
            continue
        unit = direction / length
        axes.annotate(
            "",
            xy=end - unit * 3.0,
            xytext=start + unit * 3.0,
            arrowprops={
                "arrowstyle": "-|>",
                "color": "#d62728",
                "linewidth": 2.0,
                "shrinkA": 0,
                "shrinkB": 0,
            },
            zorder=20,
        )

    axes.plot(ego_current[0], ego_current[1], marker="o", color="black", markersize=6, zorder=25)
    axes.set_xlim(ego_current[0] - radius, ego_current[0] + radius)
    axes.set_ylim(ego_current[1] - radius, ego_current[1] + radius)
    axes.set_aspect("equal")
    axes.set_title(
        title
        or "InterSim directed relations at frame %s ms (%d ego-centric edges of %d)"
        % (frame, len(window_relations), len(result.relations)),
        fontsize=10,
    )
    figure.tight_layout()
    if save_to is not None:
        Path(save_to).parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_to, dpi=140)
        print("  saved:", save_to)
    return figure

In [8]:
def rebuild_render_participants(poses, participants, step_ms, frame_ms0=0):
    """Turn the runner's pose arrays into participants the camera can render.

    The pose arrays carry positions and headings only, so the speed is recovered
    from neighbouring poses. The behavior model reads ``state.speed`` as the
    agent's initial speed, and a state without velocity would silently plan
    every agent as if it were standing still.
    """
    rebuilt = {}
    dt = step_ms / 1000.0
    for agent_id, array in poses.items():
        valid = [index for index, row in enumerate(array) if row[0] != -1.0]
        if not valid:
            continue
        source = participants.get(agent_id)
        trajectory = Trajectory(id_=agent_id, fps=round(1000.0 / step_ms, 3), stable_freq=True)
        last = None
        for index in range(valid[0], valid[-1] + 1):
            row = array[index]
            if row[0] != -1.0:
                last = (float(row[0]), float(row[1]), float(row[3]))
            speed = 0.0
            after = min(index + 1, valid[-1])
            before = max(index - 1, valid[0])
            if array[after][0] != -1.0 and array[before][0] != -1.0 and after > before:
                distance = np.hypot(
                    array[after][0] - array[before][0], array[after][1] - array[before][1]
                )
                speed = float(distance / (dt * (after - before)))
            trajectory.add_state(
                State(
                    frame=frame_ms0 + index * step_ms,
                    x=last[0],
                    y=last[1],
                    heading=last[2],
                    speed=speed,
                )
            )
        rebuilt[agent_id] = Vehicle(
            agent_id,
            "vehicle",
            trajectory=trajectory,
            length=float(getattr(source, "length", 4.8)),
            width=float(getattr(source, "width", 1.9)),
        )
        # The closed loop hands every agent its last ground-truth position as a
        # routing goal; mirror it so the re-derived plan follows the same lanes.
        if source is not None and source.trajectory.last_state is not None:
            rebuilt[agent_id].goal_xy = (
                source.trajectory.last_state.x,
                source.trajectory.last_state.y,
            )
    return rebuilt


def collect_ego_plans(model, render_participants, map_, config, ego_id, end_index, frame_ms0=0):
    """Re-derive the ego's planned trajectory at every planning frame.

    The runner commits its plans into the pose arrays but does not return them,
    so the plan trace is recovered by asking the same model, at the same frames,
    for the ego's plan on the simulated state. Over the window in which a plan
    is authoritative (up to the next replan) this reproduces what the runner
    committed.
    """
    plans = {}
    for index in range(config.planning_warmup_steps, end_index + 1, config.planning_interval):
        frame = frame_ms0 + index * config.step_ms
        result = model.plan(render_participants, map_, frame, agent_ids=[ego_id])
        trajectory = result.trajectories.get(ego_id)
        if trajectory is None:
            continue
        plans[frame] = [
            (f, trajectory.get_state(f).x, trajectory.get_state(f).y) for f in trajectory.frames
        ]
    return plans

In [9]:
def run_interactive_scenario(
    parser,
    file_name=None,
    folder=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    config=None,
    base_frame_ms=None,
    name="scenario",
    resolution=(1200, 800),
    **parse_kwargs,
):
    """Parse one scenario, run the InterSim closed loop and return its animation."""
    config = config or INTERSIM_CFG

    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )
    participants = to_lattice(participants, INTERSIM_CFG.step_ms)

    map_ = None
    if hasattr(parser, "parse_map"):
        map_ = parser.parse_map(file=file_name, folder=folder, **parse_kwargs)
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    print(f"  participants: {len(participants)},  frames: {time_range},  lanes: {len(map_.lanes)}")

    if ego_id is None:
        ego_id = tutorial_common.select_ego(participants)
    if base_frame_ms is None:
        # inD-style recordings stamp frames in absolute time, so the window
        # has to start where the ego does rather than at zero.
        base_frame_ms = int(participants[ego_id].trajectory.first_frame)
    ego = participants[ego_id]
    ego.color = tutorial_common.EGO_COLOR

    model = InterSimBehaviorModel(config)
    plan_frame = frame_at_or_after(ego, WARMUP_MS)
    result = plan_interaction_scene(model, participants, map_, plan_frame, ego_id)
    ego_edges = [e for e in result.relations if ego_id in (e.influencer, e.reactor)]
    print(
        f"  ego: {ego_id}  |  planning frame: {plan_frame}  |  modelled agents: "
        f"{len(result.scene_agent_ids)}  |  directed relations: {len(result.relations)} "
        f"({len(ego_edges)} involving the ego)"
    )
    for edge in ego_edges:
        role = "ego yields to" if edge.reactor == ego_id else "ego passes before"
        print(
            "    %s -> %s   (frame diff %s ms)  [%s %s]"
            % (
                edge.influencer,
                edge.reactor,
                edge.frame_diff,
                role,
                edge.influencer if edge.reactor == ego_id else edge.reactor,
            )
        )
    if len(result.relations) > len(ego_edges):
        print(
            "    ... and %d more edges among neighbouring vehicles"
            % (len(result.relations) - len(ego_edges))
        )
    print("  ego action:", result.actions.get(ego_id))

    plot_directed_relations(
        participants,
        map_,
        result,
        ego_id,
        plan_frame,
        title="InterSim directed relations - %s (frame %s ms)" % (name, plan_frame),
        save_to=Path("../../tests/runtime") / ("intersim_relations_%s.png" % name),
    )

    rolling = model.rollout(participants, map_, ego_id=ego_id, base_frame_ms=base_frame_ms)
    print(
        "  closed loop: collided=%s  front/side/rear=%d/%d/%d  offroad=%d  progress=%.2f m  "
        "controlled=%d"
        % (
            rolling.collided,
            rolling.front_collisions,
            rolling.side_collisions,
            rolling.rear_collisions,
            rolling.offroad_scenarios,
            rolling.progress,
            rolling.total_agents_controlled,
        )
    )

    render_participants = rebuild_render_participants(
        rolling.poses, participants, config.step_ms, frame_ms0=base_frame_ms
    )
    # The rebuilt participants are fresh Vehicle objects, so the ego colour has to
    # be re-applied to the object the renderer actually receives.
    render_participants[ego_id].color = tutorial_common.EGO_COLOR
    playback_frames = [
        base_frame_ms + index * config.step_ms
        for index in range(0, rolling.end_index + 1, PLAYBACK_STEP)
    ]
    plans = collect_ego_plans(
        model, render_participants, map_, config, ego_id, rolling.end_index, frame_ms0=base_frame_ms
    )
    print(f"  rendering {len(playback_frames)} frames, {len(plans)} re-derived ego plans ...")
    return tutorial_common.render_replay_animation(
        render_participants,
        map_,
        playback_frames,
        ego_id,
        plans=plans,
        resolution=resolution,
        fps=1000.0 / (config.step_ms * PLAYBACK_STEP),
        title_prefix="InterSim closed loop",
    )

### Example 1: WOMD - validation_interactive, Scenario 2 (the shared scene)

The junction all four behavior demos are measured on: same vehicle `8`, same frame. The relation graph has five edges here and exactly one of them involves the ego (`8 -> 6412`, the ego reaching the conflict point first), which is why the closed loop ends up controlling so few vehicles - InterSim only pushes on the agents that actually conflict.


In [10]:
ani_womd_2 = run_interactive_scenario(
    WOMDParser(),
    file_name=tutorial_common.COMPARISON_FILE,
    folder=WOMD_FOLDER,
    scenario_id=tutorial_common.COMPARISON_SCENARIO,
    ego_id=tutorial_common.COMPARISON_EGO,
    name="womd_2",
)
ani_womd_2

Parsing scenario ...
  participants: 34,  frames: (0, 8976),  lanes: 213
  ego: 8  |  planning frame: 1100  |  modelled agents: 9  |  directed relations: 5 (1 involving the ego)
    8 -> 6412   (frame diff -6 ms)  [ego passes before 6412]
    ... and 4 more edges among neighbouring vehicles
  ego action: follow
  saved: ../../tests/runtime/intersim_relations_womd_2.png
  closed loop: collided=False  front/side/rear=0/0/0  offroad=0  progress=88.60 m  controlled=1
  rendering 90 frames, 8 re-derived ego plans ...


### Example 2: WOMD - validation_interactive, Scenario 9 (the yielding case)

A second WOMD scene, picked because the ego is genuinely on the yielding side: it is almost stationary as the scenario opens, the relation graph contains `2608 -> 2478` with the ego as the reactor, and the printed action for the ego is `yield`. That is the case the model exists for - a relation that turns into actual braking rather than staying a no-op obligation.


In [11]:
ani_womd_9 = run_interactive_scenario(
    WOMDParser(),
    file_name=tutorial_common.COMPARISON_FILE,
    folder=WOMD_FOLDER,
    scenario_id=tutorial_common.SECOND_SCENARIO,
    ego_id=tutorial_common.SECOND_EGO,
    name="womd_9",
)
ani_womd_9

Parsing scenario ...
  participants: 42,  frames: (0, 9000),  lanes: 311
  ego: 2478  |  planning frame: 1100  |  modelled agents: 30  |  directed relations: 73 (2 involving the ego)
    2478 -> 2524   (frame diff 0 ms)  [ego passes before 2524]
    2608 -> 2478   (frame diff 0 ms)  [ego yields to 2608]
    ... and 71 more edges among neighbouring vehicles
  ego action: yield
  saved: ../../tests/runtime/intersim_relations_womd_9.png


  closed loop: collided=False  front/side/rear=0/0/0  offroad=0  progress=68.39 m  controlled=2
  rendering 90 frames, 8 re-derived ego plans ...


### Example 3: inD - Location 1, Recording 07 (off-domain, resampled)

A German urban junction at 25 Hz: the domain is new and the rate is new, so the runner resamples it onto the model's 100 ms lattice first. The relations still come out - `3 -> 4` here - because they are read off the geometry rather than learned, but the scene is far outside what the port was tuned on.


In [12]:
ani_ind = run_interactive_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../../data/LevelX/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
    name="ind_7",
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240)),  lanes: 137
  ego: 12  |  planning frame: 10300  |  modelled agents: 13  |  directed relations: 4 (0 involving the ego)
    ... and 4 more edges among neighbouring vehicles
  ego action: follow
  saved: ../../tests/runtime/intersim_relations_ind_7.png
  closed loop: collided=False  front/side/rear=0/0/0  offroad=0  progress=3.12 m  controlled=2
  rendering 90 frames, 8 re-derived ego plans ...


### Example 4: nuPlan - Boston Intersection (off-domain, resampled)

nuPlan sits at the other end of the map scale: one log carries a whole city's map, and the scenario is a window cut out of it. The window is the longest pass through an intersection in the log, recomputed from the log's `scenario_tag` table at run time rather than stored, because the parser stamps frames relative to `datetime(2021, 1, 1)` **in the local timezone** - a hard-coded window would mean a different stretch of road on another machine.


In [13]:
# ---- nuPlan, same four models, a city-scale map ----
NUPLAN_ROOT = "../../../data/nuplan"
nuplan_folder = f"{NUPLAN_ROOT}/data/cache/{tutorial_common.NUPLAN_SCENARIO_FOLDER}"

nuplan_parser = NuPlanParser()
nuplan_window = tutorial_common.nuplan_intersection_window(
    f"{nuplan_folder}/{tutorial_common.NUPLAN_SCENARIO_FILE}"
)
nuplan_participants, _ = nuplan_parser.parse_trajectory(
    file=tutorial_common.NUPLAN_SCENARIO_FILE, folder=nuplan_folder, time_range=nuplan_window
)
nuplan_participants = to_lattice(nuplan_participants, INTERSIM_CFG.step_ms)
nuplan_map = nuplan_parser.parse_map(
    file="map.gpkg", folder=f"{NUPLAN_ROOT}/maps/{tutorial_common.NUPLAN_SCENARIO_MAP}"
)
# The runner lays the log onto the model's 100 ms lattice itself (see
# tactics2d.behavior.rolling_utils.to_lattice).

nuplan_ego = tutorial_common.NUPLAN_SCENARIO_EGO
nuplan_participants[nuplan_ego].color = tutorial_common.EGO_COLOR
print(
    f"window {nuplan_window[0]}-{nuplan_window[1]} ms  |  {len(nuplan_participants)} participants  "
    f"|  {len(nuplan_map.lanes)} lanes"
)

nuplan_model = InterSimBehaviorModel(INTERSIM_CFG)
nuplan_rolling = nuplan_model.rollout(nuplan_participants, nuplan_map, ego_id=nuplan_ego)
print(
    "  closed loop: collided=%s  front/side/rear=%d/%d/%d  offroad=%d  progress=%.2f m  "
    "controlled=%d"
    % (
        nuplan_rolling.collided,
        nuplan_rolling.front_collisions,
        nuplan_rolling.side_collisions,
        nuplan_rolling.rear_collisions,
        nuplan_rolling.offroad_scenarios,
        nuplan_rolling.progress,
        nuplan_rolling.total_agents_controlled,
    )
)

# The log's frames are stamped in absolute time, so the replayed frames are
# offset by the window origin rather than starting at zero.
nuplan_origin = min(p.trajectory.first_frame for p in nuplan_participants.values())
render_participants = rebuild_render_participants(
    nuplan_rolling.poses, nuplan_participants, INTERSIM_CFG.step_ms, frame_ms0=nuplan_origin
)
render_participants[nuplan_ego].color = tutorial_common.EGO_COLOR
playback_frames = [
    nuplan_origin + index * INTERSIM_CFG.step_ms
    for index in range(0, nuplan_rolling.end_index + 1, PLAYBACK_STEP)
]
plans = collect_ego_plans(
    nuplan_model,
    render_participants,
    nuplan_map,
    INTERSIM_CFG,
    nuplan_ego,
    nuplan_rolling.end_index,
    frame_ms0=nuplan_origin,
)
ani_nuplan = tutorial_common.render_replay_animation(
    render_participants,
    nuplan_map,
    playback_frames,
    nuplan_ego,
    plans=plans,
    resolution=(1200, 800),
    fps=1000.0 / (INTERSIM_CFG.step_ms * PLAYBACK_STEP),
    title_prefix="InterSim closed loop (nuPlan)",
)
ani_nuplan

window 20572567849-20572584249 ms  |  100 participants  |  3019 lanes
  closed loop: collided=False  front/side/rear=0/0/0  offroad=0  progress=52.92 m  controlled=1


## Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `horizon_steps=30`, `planning_interval=20`, `interaction_distance=20` |
| **Demo defaults** | `horizon_steps=80`, `planning_interval=10`, `interaction_distance=40`, `cruise_speed=8` |
| **Wider conflict set** | `interaction_distance` decides how far a conflict is looked for, and the cost grows with the set it finds |
| **Learned direction** | `relation_mode="nn"` swaps the geometric direction for the M2I arbiter; it needs the relation checkpoint and a `decider` |
